# GulfDealFlow — Script 02: News Scraper
Scrapes Google News RSS for GCC funding announcements and extracts deal data into a staging file for review.

⚠️ **Note:** If you get 403 errors on every query, Colab's IP is being blocked. Download this notebook and run it locally instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install requests beautifulsoup4 pandas lxml -q

In [ ]:
import requests, re, os, time, pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

BASE_DIR    = '/content/drive/MyDrive/GulfDealFlow'
OUTPUT_PATH = os.path.join(BASE_DIR, 'news_staging.csv')
os.makedirs(BASE_DIR, exist_ok=True)

RSS_QUERIES = [
    'startup funding UAE million 2024',
    'startup funding Saudi Arabia million 2024',
    'venture capital GCC funding round 2024',
    'raises million seed series UAE',
    'raises million series Saudi Arabia',
    'raises million Kuwait Bahrain Oman Qatar startup',
    'startup funding UAE million 2025',
    'venture capital Saudi Arabia 2025',
]

GCC_COUNTRIES = [
    'UAE', 'United Arab Emirates', 'Dubai', 'Abu Dhabi', 'Sharjah',
    'Saudi Arabia', 'Riyadh', 'Jeddah', 'NEOM',
    'Kuwait', 'Bahrain', 'Oman', 'Muscat', 'Qatar', 'Doha'
]

STAGE_PATTERNS = {
    'Pre-Seed': r'\bpre[\-\s]?seed\b', 'Seed': r'\bseed\s+(round|funding|stage)\b',
    'Series A': r'\bseries\s+a\b', 'Series B': r'\bseries\s+b\b',
    'Series C+': r'\bseries\s+[cdefg]\b', 'Growth': r'\bgrowth\s+(round|equity)\b',
}

SECTOR_KEYWORDS = {
    'Fintech': ['fintech', 'payment', 'neobank', 'lending', 'insurtech', 'crypto', 'remittance'],
    'Proptech': ['proptech', 'real estate tech', 'property tech'],
    'Logistics & Supply Chain': ['logistics', 'supply chain', 'last mile', 'freight', 'delivery'],
    'Healthtech': ['healthtech', 'telehealth', 'medtech', 'digital health'],
    'Edtech': ['edtech', 'ed tech', 'education tech', 'e-learning'],
    'E-commerce & Retail': ['e-commerce', 'ecommerce', 'marketplace', 'retail tech'],
    'SaaS & Enterprise Software': ['saas', 'enterprise software', 'b2b software'],
    'Deep Tech & AI': ['ai startup', 'artificial intelligence', 'machine learning', 'deep tech'],
    'Energy & Cleantech': ['cleantech', 'clean energy', 'solar', 'renewable'],
    'Media & Entertainment': ['media tech', 'streaming', 'gaming', 'content platform'],
    'Food & Agritech': ['food tech', 'agritech', 'restaurant tech'],
}

def extract_amount(text):
    t = text.lower()
    for pat, mult in [
        (r'\$(\d+(?:\.\d+)?)\s*billion', 1e9), (r'\$(\d+(?:\.\d+)?)\s*million', 1e6),
        (r'(\d+(?:\.\d+)?)\s*million\s*(?:dollar|usd)', 1e6),
        (r'aed\s*(\d+(?:\.\d+)?)\s*million', 1e6/3.67),
        (r'sar\s*(\d+(?:\.\d+)?)\s*million', 1e6/3.75),
    ]:
        m = re.search(pat, t)
        if m: return int(float(m.group(1)) * mult), True
    return None, False

def extract_stage(text):
    t = text.lower()
    for stage, pat in STAGE_PATTERNS.items():
        if re.search(pat, t): return stage
    return 'Undisclosed'

def extract_country(text):
    city_map = {
        'Dubai': ('UAE','Dubai'), 'Abu Dhabi': ('UAE','Abu Dhabi'), 'Sharjah': ('UAE','Sharjah'),
        'Riyadh': ('Saudi Arabia','Riyadh'), 'Jeddah': ('Saudi Arabia','Jeddah'),
        'Muscat': ('Oman','Muscat'), 'Doha': ('Qatar','Doha')
    }
    for country in GCC_COUNTRIES:
        if country.lower() in text.lower():
            return city_map.get(country, (country, ''))
    return None, ''

def extract_sector(text):
    t = text.lower()
    for sector, kws in SECTOR_KEYWORDS.items():
        for kw in kws:
            if kw in t: return sector
    return 'Other'

def extract_company(title):
    m = re.search(r'^([A-Z][a-zA-Z0-9\s\-\.]+?)\s+(?:raises?|secures?|closes?|lands?)\s', title)
    return m.group(1).strip() if m else ''

print('Starting GulfDealFlow news scraper...\n')
all_deals, seen = [], set()
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}

for query in RSS_QUERIES:
    url = f"https://news.google.com/rss/search?q={query.replace(' ', '+')}&hl=en&gl=US&ceid=US:en"
    print(f'Querying: {query}')
    try:
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code != 200:
            print(f'  ✗ Status {r.status_code} — try running locally if this persists')
            continue
        soup = BeautifulSoup(r.text, 'xml')
        items = soup.find_all('item')
        print(f'  ✓ {len(items)} articles')
        for item in items:
            title = item.find('title').get_text(strip=True) if item.find('title') else ''
            link  = item.find('link').get_text(strip=True) if item.find('link') else ''
            pub   = item.find('pubDate').get_text(strip=True) if item.find('pubDate') else ''
            desc  = item.find('description').get_text(strip=True) if item.find('description') else ''
            if title in seen: continue
            seen.add(title)
            full = (title + ' ' + desc).lower()
            if not any(s in full for s in ['raises','secures','funding','series','seed','million','billion']): continue
            country, city = extract_country(full)
            if not country: continue
            amount, disclosed = extract_amount(full)
            try: date_str = datetime.strptime(pub[:16], '%a, %d %b %Y').strftime('%Y-%m')
            except: date_str = ''
            all_deals.append({
                'deal_id': '', 'company_name': extract_company(title),
                'country': country, 'city': city, 'date': date_str,
                'stage': extract_stage(full), 'amount_usd': amount if amount else '',
                'disclosed': 'TRUE' if disclosed else 'FALSE',
                'sector': extract_sector(full), 'description': '',
                'founded_year': '', 'website': '', 'lead_investor': '',
                'co_investors': '', 'investor_types': '', 'source': 'Google News',
                'notes': f'REVIEW | {title[:120]} | {link}'
            })
    except Exception as e:
        print(f'  ✗ Error: {e}')
    time.sleep(2)

if not all_deals:
    print('\n⚠ No deals found — likely a network block. Run this script locally.')
else:
    df = pd.DataFrame(all_deals)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f'\n✓ {len(df)} potential deals extracted')
    print(f'✓ Saved: {OUTPUT_PATH}')
    print('\n⚠ Open news_staging.csv in Drive, review every row, then run Script 03.')

In [ ]:
# Preview staging file
df.head(10)